# TabDPT Regressor — DIMER artifact inference tutorial

[GitHub](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline) · [Open in Colab](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_artifact_inference_colab.ipynb) · [Model](https://huggingface.co/Layer6/TabDPT)

**Profile:** `ARTIFACT-INFERENCE` · **DIMER Notebook Specification:** 1.0

This notebook consumes `artifact.json` + `training_context.parquet` produced **outside this execution**, validates the artifact and exact pinned base-model provenance, restores fitted preprocessing/support state, accepts genuinely new unlabelled data, predicts continuous values, and exports results. No gradient fine-tuning occurs. The release-grade path restores the repository encoder plus upstream fitted imputer/scaler/PCA state directly; it does not fit preprocessing on the uploaded artifact or inference rows.

**Trust boundary.** Digest/manifest checks establish internal consistency, not sender authenticity. This notebook accepts no ZIP, pickle, or arbitrary Python-object artifact. The context is Parquet and the exact Safetensors base weight is acquired separately and digest-verified. Use only artifacts from a trusted producer.

**Prerequisites:** externally produced artifact pair; separate CSV/Parquet new input; clean **Python 3.11+** runtime that resolves the tutorial lock set; GPU recommended/CPU slower; FlashAttention disabled for T4 portability. Uploaded data remains in the runtime; do not upload restricted data to an unauthorized environment. This tutorial does not create its own artifact, claim quality without labelled evaluation data, expose calibrated uncertainty, or support classification.


In [ ]:
from pathlib import Path
import shutil, sys
if sys.version_info < (3, 11):
    raise RuntimeError("This tutorial lock set requires Python 3.11+ (NumPy 2.3.0 upstream reproduction pin).")
REPO_DIR = Path("/content/tabdpt-regressor-pipeline")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q -r /content/tabdpt-regressor-pipeline/tutorials/requirements-colab.txt
%pip install -q --no-deps /content/tabdpt-regressor-pipeline
!git -C /content/tabdpt-regressor-pipeline rev-parse HEAD


## 1. Runtime, external artifact upload, and pre-load validation

Upload exactly `artifact.json` and `training_context.parquet`. Validation happens **before** model-state reconstruction and checks format/task, support-table path/size/SHA-256, fitted preprocessing consistency, and the manifest's base-model repository/revision/filename/digest/upstream commit against this repository contract. Because external artifact consumption is the purpose of this profile, upload is the primary path.

Release-grade artifacts include explicit format metadata and complete fitted upstream preprocessing state. Older v3 artifacts can remain compatible with repository code, but this tutorial fails closed if the loader reports that preprocessing had to use the legacy reconditioning path.


In [ ]:
import csv, io, json, platform
import importlib.metadata as md
import numpy as np, pandas as pd, torch
from google.colab import files
from tabdpt_regressor_pipeline import load_verified_artifact, validate_artifact_bundle
print("Python", platform.python_version(), "torch", md.version("torch"), "tabdpt", md.version("tabdpt"),
      "numpy", md.version("numpy"), "pandas", md.version("pandas"), "sklearn", md.version("scikit-learn"))
print("Device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "use_flash=False")
ART = Path("/content/external-tabdpt-artifact")
if ART.exists():
    shutil.rmtree(ART)
ART.mkdir(parents=True)
uploaded = files.upload()
required = {"artifact.json", "training_context.parquet"}
if set(uploaded) != required:
    raise ValueError(f"Upload exactly {sorted(required)}; got {sorted(uploaded)}")
for name, payload in uploaded.items():
    (ART / name).write_bytes(payload)
manifest_path = ART / "artifact.json"
manifest, context_path = validate_artifact_bundle(manifest_path)
encoder_state = manifest["preprocessing"]["encoder"]
expected = list(encoder_state["featureColumns"])
numeric_columns = list(encoder_state["numericColumns"])
categorical_columns = list(encoder_state["categoryMaps"].keys())
print("Validated artifact:", manifest["format"], "version", manifest.get("formatVersion", "legacy-v3-implicit"))
print(json.dumps(manifest["baseModel"], indent=2))
print({"featureCount": len(expected), "numericColumns": numeric_columns, "categoricalColumns": categorical_columns,
       "contextBytes": context_path.stat().st_size, "contextSha256": manifest["trainingContext"]["sha256"]})


## 2. Reconstruct serving state with fitted preprocessing restored

`load_verified_artifact()` resolves and digest-checks the exact pinned base weight, restores the serialized feature encoder, fitted upstream mean-imputation and standardization state, and any saved PCA basis needed for wide tables. The labelled support table is restored as TabDPT context, but the preprocessing statistics themselves are not fitted again. This is serving-state reconstruction for in-context inference, not gradient training.

The next cell also surfaces the model feature ceiling and reduction method. Feature reduction is active only when the encoded width exceeds the loaded model's `max_features`; context reduction is controlled separately by `context_size` at prediction time.


In [ ]:
pipe = load_verified_artifact(manifest_path, compile_model=False, use_flash=False)
if getattr(pipe, "preprocessing_restored_", False) is not True:
    raise RuntimeError("This artifact used the legacy compatibility/reconditioning path. Supply a release artifact with complete fitted preprocessing state.")
encoded_feature_count = len(pipe.feature_encoder.feature_columns)
model_feature_ceiling = int(pipe.estimator.max_features)
feature_reduction = str(pipe.estimator.feature_reduction)
print({"target": pipe.target_column, "encodedFeatureCount": encoded_feature_count,
       "modelFeatureCeiling": model_feature_ceiling, "featureReduction": feature_reduction,
       "featureReductionActive": encoded_feature_count > model_feature_ceiling,
       "preprocessingRestoredWithoutRefit": True})


## 3. Upload, validate, and score genuinely new input

Upload one CSV or Parquet containing exactly the fitted feature columns printed above, with no target or pre-existing `prediction`. For CSV, categorical columns are read explicitly as strings so values such as `01` are not silently converted to numbers. Parquet categorical columns are likewise cast to strings. Missing categorical values retain the fitted missing-code policy; unseen categories use the separate fitted unknown code. Numeric missing values are allowed and are transformed by the **restored training-fitted mean imputer**; non-numeric or infinite values in numeric columns fail clearly. No preprocessing is fitted on inference data.


In [ ]:
new_upload = files.upload()
if len(new_upload) != 1:
    raise ValueError("Upload exactly one CSV or Parquet input.")
input_name, raw = next(iter(new_upload.items()))
if input_name.lower().endswith(".csv"):
    text = raw.decode("utf-8-sig")
    header = next(csv.reader(io.StringIO(text)), [])
    duplicates = sorted({x for x in header if header.count(x) > 1})
    if duplicates:
        raise ValueError(f"Duplicate CSV columns: {duplicates}")
    new_data = pd.read_csv(io.BytesIO(raw), dtype={c: "string" for c in categorical_columns if c in header})
elif input_name.lower().endswith((".parquet", ".pq")):
    new_data = pd.read_parquet(io.BytesIO(raw), engine="pyarrow")
    for column in categorical_columns:
        if column in new_data.columns:
            new_data[column] = new_data[column].astype("string")
else:
    raise ValueError("Input must be CSV or Parquet.")
if new_data.columns.duplicated().any():
    raise ValueError("Duplicate columns are not supported.")
if pipe.target_column in new_data or "prediction" in new_data:
    raise ValueError("Remove target/prediction columns before inference.")
missing = [x for x in expected if x not in new_data]
extra = [x for x in new_data if x not in expected]
if missing or extra:
    raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")
new_data = new_data.loc[:, expected].copy()
for column in numeric_columns:
    original = new_data[column]
    converted = pd.to_numeric(original, errors="coerce")
    invalid = original.notna() & converted.isna()
    if invalid.any():
        raise ValueError(f"Numeric feature {column!r} contains non-numeric values.")
    finite = converted.dropna().to_numpy(dtype=float)
    if finite.size and not np.isfinite(finite).all():
        raise ValueError(f"Numeric feature {column!r} contains infinite values.")
    new_data[column] = converted
missing_counts = new_data.isna().sum()
print("Input rows/features:", new_data.shape, "missing:", missing_counts[missing_counts > 0].to_dict() or "none")
kw = {"n_ensembles": 2, "context_size": 512, "batch_size": 512, "seed": 42}
context_rows = len(pd.read_parquet(context_path, engine="pyarrow"))
print("Requested context_size:", kw["context_size"], "effective support rows <=", min(context_rows, kw["context_size"]))
pred = pipe.predict(new_data, **kw)
results = pd.DataFrame({"row_id": new_data.index.to_numpy(), "prediction": pred.to_numpy()})
results.head()


## 4. Export predictions and provenance

Predictions are continuous point estimates, not calibrated uncertainty intervals. `row_id` maps each prediction to its input. Provenance records the externally supplied artifact identity, immutable model contract, runtime, restored preprocessing/capacity state, inference configuration, and input shape; it contains no credentials.


In [ ]:
OUT = Path("/content/tabdpt-artifact-inference-output")
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / "tabdpt_regression_predictions.csv", index=False)
provenance = {
    "artifact": {"format": manifest["format"], "formatVersion": manifest.get("formatVersion"),
                 "baseModel": manifest["baseModel"], "trainingContextSha256": manifest["trainingContext"]["sha256"]},
    "runtime": {"python": platform.python_version(), "torch": md.version("torch"), "tabdpt": md.version("tabdpt"),
                "numpy": md.version("numpy"), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
                "use_flash": False},
    "preprocessing": {"preprocessingRestoredWithoutRefit": True, "encodedFeatureCount": encoded_feature_count,
                      "modelFeatureCeiling": model_feature_ceiling, "featureReduction": feature_reduction,
                      "featureReductionActive": encoded_feature_count > model_feature_ceiling,
                      "numericMissingPolicy": "restored training-fitted mean imputation",
                      "categoricalMissingPolicy": "restored fitted missing code",
                      "unknownCategoryPolicy": "restored fitted unknown code"},
    "inference": {**kw, "artifactContextRows": context_rows, "effectiveSupportRowsAtMost": min(context_rows, kw["context_size"])},
    "input": {"filename": input_name, "rows": len(new_data), "features": list(new_data.columns)},
}
(OUT / "tabdpt_regression_inference_provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
print("Wrote prediction CSV and provenance JSON.")


## Interpretation and troubleshooting

A successful run proves an independently supplied release artifact is internally consistent with this repository's pinned model contract, restores fitted preprocessing without refitting it, reconstructs support context, and scores schema-compatible new records with explicit capacity/context behavior. It does **not** authenticate the producer or establish predictive quality, robustness, calibration, fairness, or production fitness. Never bypass a failed manifest, digest, preprocessing-state, or schema check; obtain a correct trusted artifact. If base-model acquisition fails, use only the exact trusted `tabdpt1_2.safetensors` matching the repository/manifest digest rather than substituting another model.
